# ST-CDGM — Présentation globale des résultats (checkpoint + graphiques)

Notebook orienté **figures** : résumé quantitatif, cartes, distributions, spectre, matrice DAG.

- Réutilise le chargement checkpoint / modèles comme `st_cdgm_validation_inference.ipynb`.
- Métriques agrégées : `stats.md`, `results/validation/*.csv` / `validation_report.json` si présents.
- Si `data/raw/test/` est disponible, une inférence de démonstration produit cartes et histogrammes.


In [ ]:
import os
import sys
import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
from omegaconf import OmegaConf

warnings.filterwarnings("ignore", category=UserWarning)

project_root = Path.cwd()
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

CONFIG = OmegaConf.load("config/training_config.yaml")
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() and CONFIG.training.device == "cuda" else "cpu"
)
sns.set_theme(style="whitegrid", context="notebook")
print(f"Device: {DEVICE}")


In [ ]:
from st_cdgm.data.pipeline import NetCDFDataPipeline
from st_cdgm.models.graph_builder import HeteroGraphBuilder
from st_cdgm.models.intelligible_encoder import IntelligibleVariableConfig, IntelligibleVariableEncoder
from st_cdgm.models.causal_rcn import RCNCell, RCNSequenceRunner
from st_cdgm.models.diffusion_decoder import CausalDiffusionDecoder
# Metriques disponibles via st_cdgm.evaluation.evaluation_xai si besoin (evaluate_metrics, plot_dag_heatmap)

USER_CHECKPOINT_PATH = None
_env_ckpt = os.environ.get("ST_CDGM_CHECKPOINT", "").strip()
if _env_ckpt:
    USER_CHECKPOINT_PATH = Path(_env_ckpt)

_save_dir = Path(CONFIG.checkpoint.get("save_dir", "models"))
_candidates = []
if USER_CHECKPOINT_PATH:
    _candidates.append(Path(USER_CHECKPOINT_PATH))
for name in ("st_cdgm_checkpoint_best.pth", "st_cdgm_checkpoint.pth", "st_cdgm_checkpoint_last.pth"):
    _candidates.append(_save_dir / name)

CHECKPOINT_PATH = next((p for p in _candidates if p.is_file()), None)
if CHECKPOINT_PATH is None and _save_dir.is_dir():
    _pth = sorted(_save_dir.glob("*.pth"), key=lambda p: p.stat().st_mtime, reverse=True)
    if _pth:
        CHECKPOINT_PATH = _pth[0]

if CHECKPOINT_PATH is None or not CHECKPOINT_PATH.is_file():
    raise FileNotFoundError("Aucun checkpoint .pth trouvé dans models/")

try:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
except TypeError:
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE)

for k in ("encoder_state_dict", "rcn_cell_state_dict", "diffusion_state_dict"):
    if k not in checkpoint:
        raise KeyError(f"Clé manquante: {k}")

print("Checkpoint:", CHECKPOINT_PATH)
print("Epoch:", checkpoint.get("epoch", "n/a"))


## Configuration test (GCM) et reconstruction du modèle


In [ ]:
TEST_ROOT = Path("data/raw/test")
TEST_GCMS = []


def test_lr_path(gcm):
    return str(TEST_ROOT / f"{gcm}_histupdated_compressed.nc")


def test_hr_path(gcm):
    return str(TEST_ROOT / f"{gcm}_historical_precip_compressed.nc")


def discover_test_gcms():
    if not TEST_ROOT.is_dir():
        return []
    lr_suffix = "_histupdated_compressed.nc"
    hr_suffix = "_historical_precip_compressed.nc"
    lr_stems = {f.name.replace(lr_suffix, "") for f in TEST_ROOT.glob(f"*{lr_suffix}")}
    hr_stems = {f.name.replace(hr_suffix, "") for f in TEST_ROOT.glob(f"*{hr_suffix}")}
    return sorted(lr_stems.intersection(hr_stems))


TEST_GCMS = discover_test_gcms()
HAS_TEST_DATA = len(TEST_GCMS) > 0
print("GCMs test:", TEST_GCMS if HAS_TEST_DATA else "(aucun — figures inférence désactivées)")


In [ ]:
# Rebuild pipeline + builder + modèles depuis checkpoint (cf. st_cdgm_validation_inference.ipynb)
ckpt_cfg = checkpoint.get("config", {})
ckpt_cfg_full = checkpoint.get("config_full", {}) or {}

SEQ_LEN = int(ckpt_cfg.get("seq_len", CONFIG.data.seq_len))
STATIC_PATH = str(CONFIG.data.static_path) if CONFIG.data.get("static_path") else None

norm_cfg = ckpt_cfg.get("normalization", {})
MEAN_PATH = str(norm_cfg.get("mean_path", "data/raw/normalization_coefs/mean_1974_2011.nc"))
STD_PATH = str(norm_cfg.get("std_path", "data/raw/normalization_coefs/std_1974_2011.nc"))

LR_VARIABLES = list(ckpt_cfg.get("lr_variables", CONFIG.data.lr_variables))
STATIC_VARIABLES = list(ckpt_cfg.get("static_variables", CONFIG.data.static_variables))
TEST_HR_VARIABLES = ["pr"]

sampling_cfg = ckpt_cfg.get("sampling", {})
# Forcer 8 pas (train_timesteps / sampling) — ignore les configs YAML et checkpoint pour éviter
# l'erreur diffusers : num_inference_steps > num_train_timesteps (ex. 15 vs 8).
NOTEBOOK_DIFFUSION_STEPS = 8
EVAL_NUM_STEPS = NOTEBOOK_DIFFUSION_STEPS
EVAL_SCHEDULER = sampling_cfg.get("scheduler_type", CONFIG.diffusion.get("scheduler_type", "ddpm"))
EVAL_NUM_SAMPLES = int(CONFIG.evaluation.get("num_samples", 5)) if CONFIG.get("evaluation") else 5

_num_train = NOTEBOOK_DIFFUSION_STEPS


def build_test_pipeline(gcm):
    return NetCDFDataPipeline(
        lr_path=test_lr_path(gcm),
        hr_path=test_hr_path(gcm),
        static_path=STATIC_PATH,
        seq_len=SEQ_LEN,
        baseline_strategy=CONFIG.data.baseline_strategy,
        baseline_factor=CONFIG.data.get("baseline_factor", 4),
        normalize=bool(CONFIG.data.normalize),
        nan_fill_strategy=CONFIG.data.nan_fill_strategy,
        precipitation_delta=CONFIG.data.get("precipitation_delta", 0.01),
        lr_variables=LR_VARIABLES,
        hr_variables=TEST_HR_VARIABLES,
        static_variables=STATIC_VARIABLES,
        means_path=MEAN_PATH if Path(MEAN_PATH).exists() else None,
        stds_path=STD_PATH if Path(STD_PATH).exists() else None,
    )


probe_pipeline = None
probe_sample = None
builder = None
encoder = None
rcn_cell = None
rcn_runner = None
diffusion = None
VAR_NAMES_DAG = []

if HAS_TEST_DATA and len(TEST_GCMS) > 0:
    probe_gcm = TEST_GCMS[0]
    probe_pipeline = build_test_pipeline(probe_gcm)
    probe_dataset = probe_pipeline.build_sequence_dataset(seq_len=SEQ_LEN, as_torch=True)
    probe_sample = next(iter(probe_dataset))

    lr_shape = tuple(ckpt_cfg.get("lr_shape", CONFIG.graph.lr_shape))
    hr_shape = tuple(ckpt_cfg.get("hr_shape", CONFIG.graph.hr_shape))
    include_mid_layer = bool(ckpt_cfg_full.get("graph", {}).get("include_mid_layer", CONFIG.graph.include_mid_layer))

    builder = HeteroGraphBuilder(
        lr_shape=lr_shape,
        hr_shape=hr_shape,
        static_dataset=probe_pipeline.get_static_dataset(),
        include_mid_layer=include_mid_layer,
    )

    allowed_nodes = set(builder.dynamic_node_types) | set(builder.static_node_types)
    encoder_configs = [
        IntelligibleVariableConfig(
            name=mp.name,
            meta_path=(mp.src, mp.relation, mp.target),
            pool=mp.get("pool", "mean"),
        )
        for mp in CONFIG.encoder.metapaths
        if mp.src in allowed_nodes and mp.target in allowed_nodes
    ]
    if probe_pipeline.get_static_dataset() is not None:
        encoder_configs.append(
            IntelligibleVariableConfig(
                name="static",
                meta_path=("SP_HR", "causes", "GP850"),
                pool="mean",
            )
        )

    hr_channels = probe_sample["residual"].shape[1]
    rcn_driver_dim = probe_sample["lr"].shape[1]

    encoder = IntelligibleVariableEncoder(
        configs=encoder_configs,
        hidden_dim=CONFIG.encoder.hidden_dim,
        conditioning_dim=CONFIG.encoder.conditioning_dim,
    ).to(DEVICE)

    rcn_cell = RCNCell(
        num_vars=len(encoder_configs),
        hidden_dim=CONFIG.rcn.hidden_dim,
        driver_dim=rcn_driver_dim,
        reconstruction_dim=rcn_driver_dim,
        dropout=CONFIG.rcn.dropout,
    ).to(DEVICE)

    rcn_runner = RCNSequenceRunner(rcn_cell, detach_interval=CONFIG.rcn.get("detach_interval"))

    diffusion = CausalDiffusionDecoder(
        in_channels=hr_channels,
        conditioning_dim=CONFIG.diffusion.conditioning_dim,
        height=CONFIG.diffusion.height,
        width=CONFIG.diffusion.width,
        num_diffusion_steps=_num_train,
        unet_kwargs=dict(
            layers_per_block=1,
            block_out_channels=(32, 64),
            down_block_types=("DownBlock2D", "CrossAttnDownBlock2D"),
            up_block_types=("CrossAttnUpBlock2D", "UpBlock2D"),
            mid_block_type="UNetMidBlock2D",
            norm_num_groups=8,
            class_embed_type="projection",
            projection_class_embeddings_input_dim=len(encoder_configs) * CONFIG.diffusion.conditioning_dim,
            resnet_time_scale_shift="scale_shift",
            attention_head_dim=32,
            only_cross_attention=[False, True],
        ),
    ).to(DEVICE)

    try:
        from st_cdgm.utils.checkpoint import strip_torch_compile_prefix
    except ModuleNotFoundError:
        def strip_torch_compile_prefix(sd):
            if not sd:
                return sd
            if not any(str(k).startswith("_orig_mod.") for k in sd.keys()):
                return sd
            return {
                (k[len("_orig_mod."):] if str(k).startswith("_orig_mod.") else k): v
                for k, v in sd.items()
            }

    encoder.load_state_dict(strip_torch_compile_prefix(checkpoint["encoder_state_dict"]))
    rcn_cell.load_state_dict(strip_torch_compile_prefix(checkpoint["rcn_cell_state_dict"]))
    diffusion.load_state_dict(strip_torch_compile_prefix(checkpoint["diffusion_state_dict"]))

    encoder.eval()
    rcn_runner.cell.eval()
    diffusion.eval()

    # DDPM : num_inference_steps <= num_train_timesteps
    EVAL_NUM_STEPS = min(int(EVAL_NUM_STEPS), int(diffusion.num_diffusion_steps))

    VAR_NAMES_DAG = [cfg.name for cfg in encoder_configs]
    print("Modèles chargés. Sampling:", EVAL_NUM_STEPS, EVAL_SCHEDULER, "samples:", EVAL_NUM_SAMPLES)
else:
    print(
        "Pas de données test (data/raw/test) — modèles non reconstruits. "
        "Ajoutez des paires LR/HR pour l'inférence et les figures."
    )


## Tableau de bord : métriques depuis fichiers


In [ ]:
def load_metrics_dataframe():
    p_csv = Path("results/validation/validation_metrics_by_gcm.csv")
    if p_csv.is_file():
        return pd.read_csv(p_csv), "csv"
    p_json = Path("results/validation/validation_report.json")
    if p_json.is_file():
        with open(p_json, encoding="utf-8") as f:
            rep = json.load(f)
        gs = rep.get("global_summary", {})
        return pd.DataFrame([gs]), "json_global"
    return None, None


metrics_df, metrics_source = load_metrics_dataframe()
print("Source métriques:", metrics_source)
if metrics_df is not None:
    try:
        from IPython.display import display
        display(metrics_df.head(20))
    except Exception:
        print(metrics_df.head(20))
else:
    print("Aucun CSV/JSON — exécuter st_cdgm_validation_inference ou ajouter results/validation/")


## Barres : modèle vs baseline (colonnes disponibles)


In [ ]:
if metrics_df is not None and not metrics_df.empty:
    plot_cols = [c for c in ("MSE", "MAE", "crps", "F1_p95", "F1_p99") if c in metrics_df.columns]
    if plot_cols and "gcm" in metrics_df.columns:
        fig, axes = plt.subplots(1, len(plot_cols), figsize=(4 * len(plot_cols), 4))
        if len(plot_cols) == 1:
            axes = [axes]
        for ax, col in zip(axes, plot_cols):
            metrics_df.plot(x="gcm", y=col, kind="bar", ax=ax, legend=False)
            ax.set_title(col)
        plt.tight_layout()
        plt.show()
    elif plot_cols:
        metrics_df[plot_cols].mean().plot(kind="bar", figsize=(8, 4))
        plt.title("Moyenne (colonnes numériques)")
        plt.tight_layout()
        plt.show()
else:
    print("Pas de barres — DataFrame vide.")


## Cartes par cas : LR / HR observé / prédiction HR

Trois cellules ci‑dessous : un cas par `skip` dans le flux de fenêtres (`CASE_SKIPS_PRESENT`). Avec Cartopy : chaque sous-graphique utilise `plot_field_on_map` et `set_map_extent_from_data(lon, lat)` (coordonnées des NetCDF du pipeline), comme [`st_cdgm_publication_figures.ipynb`](st_cdgm_publication_figures.ipynb) ; sans Cartopy, repli sur `imshow` et les mêmes extents. **Tailles et ratio** : une carte `9×7`, `n` cartes en ligne `(9n)×7`, projection `PlateCarree(central_longitude=171.77)` ; heatmap DAG : ratio carte NZ `12×10`.


In [ ]:
try:
    import cartopy.crs as ccrs
    HAS_CARTOPY = True
except ImportError:
    ccrs = None
    HAS_CARTOPY = False

import xarray as xr

CASE_SKIPS_PRESENT = [0, 25, 75]
FUTURE_SKIPS = [120, 180, 240]
LR_PLOT_CHANNEL = 0

# Réf. st_cdgm_publication_figures : new_map_figure(figsize=(9, 7)), carte NZ (12, 10)
_MAP_SINGLE_W, _MAP_SINGLE_H = 9, 7
_MAP_NZ_W, _MAP_NZ_H = 12, 10

# Module cartopy.crs (comme publication : plot_field_on_map / set_map_extent_from_data)
ccrs_m = ccrs if HAS_CARTOPY else None


def _cartopy_figsize_ncols(n_cols):
    """Ratio publication : une carte = 9×7 ; n cartes côte à côte = (9*n)×7."""
    return (_MAP_SINGLE_W * n_cols, _MAP_SINGLE_H)


def _cartopy_data_crs():
    """PlateCarree(central_longitude=171.77) comme new_map_figure / carte NZ."""
    if not HAS_CARTOPY:
        return None
    return ccrs.PlateCarree(central_longitude=171.77)


def _map_caption_lines(sample, gcm_name=None, lr_var_name=None):
    lines = []
    if sample is not None:
        t = sample.get("time")
        if t is not None:
            try:
                le = len(t)
            except TypeError:
                le = 0
            if le > 0:
                lines.append(f"Instant cible (dernier pas) : {t[-1]}")
                if le > 1:
                    lines.append(f"Début fenêtre : {t[0]}  |  Longueur séquence : {le}")
    if gcm_name:
        lines.append(f"GCM test : {gcm_name}")
    if lr_var_name:
        lines.append(f"Canal LR cartographié : {lr_var_name}")
    return "\n".join(lines)


def apply_suptitle_safe(fig, title, y_cartopy=1.02):
    """Préférer `figure_title=` dans plot_row_lr_hr_pred pour les cartes Cartopy (layout cohérent)."""
    if HAS_CARTOPY:
        fig.suptitle(title, fontsize=12, y=y_cartopy)
        eng = fig.get_layout_engine()
        if eng is not None:
            try:
                eng.execute(fig)
            except Exception:
                pass
    else:
        fig.suptitle(title, fontsize=12)
        plt.tight_layout()


def lon_lat_1d_from_pipe(pipe, grid):
    """Longitudes et latitudes 1D des NetCDF du pipeline (lots LR ou HR)."""
    ds = pipe.get_hr_dataset() if grid == "hr" else pipe.get_lr_dataset()
    lat_name = pipe.dims.hr_lat if grid == "hr" else pipe.dims.lr_lat
    lon_name = pipe.dims.hr_lon if grid == "hr" else pipe.dims.lr_lon
    lon1d = np.asarray(ds[lon_name].values, dtype=float)
    lat1d = np.asarray(ds[lat_name].values, dtype=float)
    return lon1d, lat1d


def numpy_field_to_dataarray(field2d, lon1d, lat1d):
    """Champ 2D (lat, lon) + coordonnées — pour plot_field_on_map (xarray), après align_raster_lower_origin."""
    return xr.DataArray(
        field2d,
        dims=("lat", "lon"),
        coords={"lat": ("lat", lat1d), "lon": ("lon", lon1d)},
    )


def plot_field_on_map(da, ax, ccrs_mod, cmap, *, robust=True, norm=None):
    """Comme st_cdgm_publication_figures : pcolormesh + transform PlateCarree + coastlines."""
    if ccrs_mod is None:
        return da.plot(ax=ax, cmap=cmap, robust=robust, norm=norm, add_colorbar=False)
    kw = dict(ax=ax, cmap=cmap, transform=ccrs_mod.PlateCarree(), add_colorbar=False)
    if norm is not None:
        kw["norm"] = norm
        kw["robust"] = False
    else:
        kw["robust"] = robust
    im = da.plot(**kw)
    ax.coastlines(resolution="50m", zorder=4)
    return im


def set_map_extent_from_data(ax, ccrs_mod, lon1d, lat1d, pad_deg=2.0):
    """Cadre carte autour des données (Cartopy uniquement)."""
    if ccrs_mod is None:
        return
    lo = float(np.nanmin(lon1d))
    hi = float(np.nanmax(lon1d))
    la0 = float(np.nanmin(lat1d))
    la1 = float(np.nanmax(lat1d))
    ax.set_extent(
        [lo - pad_deg, hi + pad_deg, la0 - pad_deg, la1 + pad_deg],
        crs=ccrs_mod.PlateCarree(),
    )


def _extent_from_ds(ds, lat_name, lon_name):
    lat = np.asarray(ds[lat_name].values, dtype=float)
    lon = np.asarray(ds[lon_name].values, dtype=float)
    lat_min, lat_max = float(np.nanmin(lat)), float(np.nanmax(lat))
    lon_min, lon_max = float(np.nanmin(lon)), float(np.nanmax(lon))
    return [lon_min, lon_max, lat_min, lat_max]


def extent_hr(pipe):
    return _extent_from_ds(pipe.get_hr_dataset(), pipe.dims.hr_lat, pipe.dims.hr_lon)


def extent_lr(pipe):
    return _extent_from_ds(pipe.get_lr_dataset(), pipe.dims.lr_lat, pipe.dims.lr_lon)


def align_raster_lower_origin(field2d, pipe, grid):
    ds = pipe.get_hr_dataset() if grid == "hr" else pipe.get_lr_dataset()
    lat_name = pipe.dims.hr_lat if grid == "hr" else pipe.dims.lr_lat
    lat = np.asarray(ds[lat_name].values, dtype=float)
    arr = np.asarray(field2d, dtype=float)
    if lat.size >= 2 and lat[0] > lat[-1]:
        return np.flipud(arr)
    return arr


def nth_sample(iterable_dataset, n):
    it = iter(iterable_dataset)
    sample = None
    for i in range(n + 1):
        try:
            sample = next(it)
        except StopIteration:
            return None, i
    return sample, n


def lr_map_2d(sample, channel=LR_PLOT_CHANNEL):
    lr = sample["lr"][-1]
    if hasattr(lr, "numpy"):
        lr = lr.numpy()
    return np.asarray(lr[channel], dtype=float)


@torch.no_grad()
def hr_prediction_explain_bundle(sample):
    """Inférence + éléments pour explicabilité MVP : DAG, baseline, résidu, correction prédite."""
    samples_out, target_batch, baseline_batch, dag_last = run_inference_with_dag(sample)
    pred_stack = torch.stack([s.t_mean for s in samples_out], dim=0).cpu()
    pred_mean = pred_stack.mean(dim=0).squeeze(0).numpy()
    tgt = target_batch.squeeze(0).numpy()
    if tgt.ndim == 3:
        tgt = tgt[0]
    if pred_mean.ndim == 3:
        pred_mean = pred_mean[0]
    bl = baseline_batch.squeeze(0).cpu().numpy()
    if bl.ndim == 3:
        bl = bl[0]
    res_t = sample["residual"][-1]
    if hasattr(res_t, "numpy"):
        res_t = res_t.numpy()
    res_t = np.asarray(res_t, dtype=float)
    if res_t.ndim == 3:
        res_t = res_t[0]
    pred_corr = pred_mean - bl
    return {
        "pred": pred_mean,
        "tgt": tgt,
        "baseline": bl,
        "residual_true": res_t,
        "pred_correction": pred_corr,
        "dag_last": dag_last,
    }


def hr_pred_and_target(sample):
    b = hr_prediction_explain_bundle(sample)
    return b["pred"], b["tgt"]


def plot_residual_decomposition(
    pipe, baseline, resid_true, pred_corr, title_main, *, sample=None, gcm_name=None
):
    """Y = Baseline + résidu ; correction IA ≈ prédiction − Baseline (cf. explicabilite.md)."""
    from matplotlib.colors import TwoSlopeNorm

    ex_hr = extent_hr(pipe)
    b = align_raster_lower_origin(baseline, pipe, "hr")
    r = align_raster_lower_origin(resid_true, pipe, "hr")
    c = align_raster_lower_origin(pred_corr, pipe, "hr")
    finite = np.isfinite(r) & np.isfinite(c)
    if np.any(finite):
        vmax = float(
            max(np.nanmax(np.abs(r[finite])), np.nanmax(np.abs(c[finite])), 1e-12)
        )
    else:
        vmax = 1.0
    norm_div = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)
    cap = _map_caption_lines(sample, gcm_name)
    if HAS_CARTOPY:
        lon1d, lat1d = lon_lat_1d_from_pipe(pipe, "hr")
        da_b = numpy_field_to_dataarray(b, lon1d, lat1d)
        da_r = numpy_field_to_dataarray(r, lon1d, lat1d)
        da_c = numpy_field_to_dataarray(c, lon1d, lat1d)
        fig, axes = plt.subplots(
            1,
            3,
            figsize=_cartopy_figsize_ncols(3),
            subplot_kw={"projection": _cartopy_data_crs()},
            layout="constrained",
        )
        ax1, ax2, ax3 = axes[0], axes[1], axes[2]
        im1 = plot_field_on_map(da_b, ax1, ccrs_m, "Blues", robust=False)
        set_map_extent_from_data(ax1, ccrs_m, lon1d, lat1d)
        ax1.set_title("Baseline HR (déterministe)")
        im2 = plot_field_on_map(da_r, ax2, ccrs_m, "coolwarm", robust=False, norm=norm_div)
        set_map_extent_from_data(ax2, ccrs_m, lon1d, lat1d)
        ax2.set_title("Résidu vrai (cible)")
        im3 = plot_field_on_map(da_c, ax3, ccrs_m, "coolwarm", robust=False, norm=norm_div)
        set_map_extent_from_data(ax3, ccrs_m, lon1d, lat1d)
        ax3.set_title("Correction prédite (préd − baseline)")
        fig.colorbar(im1, ax=ax1, shrink=0.75, pad=0.02)
        fig.colorbar(im2, ax=ax2, shrink=0.75, pad=0.02)
        fig.colorbar(im3, ax=ax3, shrink=0.75, pad=0.02)
        if cap:
            fig.supxlabel(cap, fontsize=9)
        fig.suptitle(title_main, fontsize=12, y=1.0)
    else:
        fig, axes = plt.subplots(1, 3, figsize=_cartopy_figsize_ncols(3))
        axes[0].imshow(b, origin="lower", extent=ex_hr, cmap="Blues")
        axes[0].set_title("Baseline HR (déterministe)")
        im2 = axes[1].imshow(
            r, origin="lower", extent=ex_hr, cmap="coolwarm", norm=norm_div
        )
        axes[1].set_title("Résidu vrai (cible)")
        axes[2].imshow(c, origin="lower", extent=ex_hr, cmap="coolwarm", norm=norm_div)
        axes[2].set_title("Correction prédite (préd − baseline)")
        fig.colorbar(im2, ax=axes.ravel().tolist(), shrink=0.7)
        if cap:
            fig.supxlabel(cap, fontsize=9)
        fig.suptitle(title_main)
        plt.tight_layout()
    return fig


def plot_dag_explain(A_tensor, title_suffix="", *, sample=None, gcm_name=None):
    """Heatmap DAG instantané (RCN), aligné sur explicabilite.md (structure causale)."""
    A = A_tensor.detach().cpu()
    if A.dim() == 3:
        A = A.mean(dim=0)
    A_np = A.numpy()
    n = A_np.shape[0]
    names = (
        VAR_NAMES_DAG
        if len(VAR_NAMES_DAG) == n
        else [f"V{i}" for i in range(n)]
    )
    # Ratio publication carte NZ (12, 10) — échelle selon n
    scale = max(0.65, min(1.2, 0.5 + 0.06 * float(n)))
    fig_w = _MAP_NZ_W * scale
    fig_h = _MAP_NZ_H * scale
    fig, ax = plt.subplots(figsize=(fig_w, fig_h), layout="constrained")
    sns.heatmap(
        A_np,
        xticklabels=names,
        yticklabels=names,
        cmap="coolwarm",
        center=0.0,
        ax=ax,
    )
    ax.set_title(f"Matrice DAG (dernier pas RCN) — {title_suffix}")
    cap = _map_caption_lines(sample, gcm_name)
    if cap:
        fig.supxlabel(cap, fontsize=9)
    return fig


def plot_row_lr_hr_pred(
    pipe, sample, pred_mean, tgt, lr_name=None, gcm_name=None, figure_title=None
):
    ex_lr = extent_lr(pipe)
    ex_hr = extent_hr(pipe)
    lr_d = align_raster_lower_origin(lr_map_2d(sample), pipe, "lr")
    hr_t = align_raster_lower_origin(tgt, pipe, "hr")
    hr_p = align_raster_lower_origin(pred_mean, pipe, "hr")
    lr_lbl = f"LR ({lr_name or 'canal ' + str(LR_PLOT_CHANNEL)})"
    cap = _map_caption_lines(sample, gcm_name, lr_var_name=lr_name)
    if HAS_CARTOPY:
        lon_lr, lat_lr = lon_lat_1d_from_pipe(pipe, "lr")
        lon_hr, lat_hr = lon_lat_1d_from_pipe(pipe, "hr")
        da_lr = numpy_field_to_dataarray(lr_d, lon_lr, lat_lr)
        da_ht = numpy_field_to_dataarray(hr_t, lon_hr, lat_hr)
        da_hp = numpy_field_to_dataarray(hr_p, lon_hr, lat_hr)
        fig, axes = plt.subplots(
            1,
            3,
            figsize=_cartopy_figsize_ncols(3),
            subplot_kw={"projection": _cartopy_data_crs()},
            layout="constrained",
        )
        ax1, ax2, ax3 = axes[0], axes[1], axes[2]
        im1 = plot_field_on_map(da_lr, ax1, ccrs_m, "viridis", robust=False)
        set_map_extent_from_data(ax1, ccrs_m, lon_lr, lat_lr)
        ax1.set_title(lr_lbl)
        im2 = plot_field_on_map(da_ht, ax2, ccrs_m, "Blues", robust=False)
        set_map_extent_from_data(ax2, ccrs_m, lon_hr, lat_hr)
        ax2.set_title("HR observé (dernier pas)")
        im3 = plot_field_on_map(da_hp, ax3, ccrs_m, "Blues", robust=False)
        set_map_extent_from_data(ax3, ccrs_m, lon_hr, lat_hr)
        ax3.set_title("HR prédit (moy. ensemble)")
        fig.colorbar(im1, ax=ax1, shrink=0.75, pad=0.02)
        fig.colorbar(im2, ax=ax2, shrink=0.75, pad=0.02)
        fig.colorbar(im3, ax=ax3, shrink=0.75, pad=0.02)
        if cap:
            fig.supxlabel(cap, fontsize=9)
        if figure_title:
            fig.suptitle(figure_title, fontsize=12, y=1.0)
    else:
        fig, axes = plt.subplots(1, 3, figsize=_cartopy_figsize_ncols(3))
        axes[0].imshow(lr_d, origin="lower", extent=ex_lr, cmap="viridis")
        axes[0].set_title(lr_lbl)
        axes[1].imshow(hr_t, origin="lower", extent=ex_hr, cmap="Blues")
        axes[1].set_title("HR observé (dernier pas)")
        axes[2].imshow(hr_p, origin="lower", extent=ex_hr, cmap="Blues")
        axes[2].set_title("HR prédit (moy. ensemble)")
        if cap:
            fig.supxlabel(cap, fontsize=9)
        if figure_title:
            fig.suptitle(figure_title, fontsize=12)
        plt.tight_layout()
    return fig


def plot_pred_vs_obs(pipe, pred, obs, title_main, *, sample=None, gcm_name=None):
    from matplotlib.cm import ScalarMappable
    from matplotlib.colors import Normalize

    ex_hr = extent_hr(pipe)
    p = align_raster_lower_origin(pred, pipe, "hr")
    o = align_raster_lower_origin(obs, pipe, "hr")
    finite = np.isfinite(p) & np.isfinite(o)
    if not np.any(finite):
        vmin, vmax = 0.0, 1.0
    else:
        vmin = float(np.nanmin([np.nanmin(p[finite]), np.nanmin(o[finite])]))
        vmax = float(np.nanmax([np.nanmax(p[finite]), np.nanmax(o[finite])]))
    if vmin >= vmax:
        vmax = vmin + 1e-6
    norm = Normalize(vmin=vmin, vmax=vmax)
    cap = _map_caption_lines(sample, gcm_name)
    if HAS_CARTOPY:
        lon1d, lat1d = lon_lat_1d_from_pipe(pipe, "hr")
        da_p = numpy_field_to_dataarray(p, lon1d, lat1d)
        da_o = numpy_field_to_dataarray(o, lon1d, lat1d)
        fig, axes = plt.subplots(
            1,
            2,
            figsize=_cartopy_figsize_ncols(2),
            subplot_kw={"projection": _cartopy_data_crs()},
            layout="constrained",
        )
        ax1, ax2 = axes[0], axes[1]
        plot_field_on_map(da_p, ax1, ccrs_m, "Blues", robust=False, norm=norm)
        set_map_extent_from_data(ax1, ccrs_m, lon1d, lat1d)
        ax1.set_title("Prédit")
        plot_field_on_map(da_o, ax2, ccrs_m, "Blues", robust=False, norm=norm)
        set_map_extent_from_data(ax2, ccrs_m, lon1d, lat1d)
        ax2.set_title("Observé")
        sm = ScalarMappable(norm=norm, cmap="Blues")
        sm.set_array([])
        fig.colorbar(sm, ax=[ax1, ax2], shrink=0.75, pad=0.02, label="pr (transformée)")
        if cap:
            fig.supxlabel(cap, fontsize=9)
        fig.suptitle(title_main, fontsize=12, y=1.0)
    else:
        fig, axes = plt.subplots(1, 2, figsize=_cartopy_figsize_ncols(2))
        im0 = axes[0].imshow(p, origin="lower", extent=ex_hr, cmap="Blues", norm=norm)
        axes[0].set_title("Prédit")
        axes[1].imshow(o, origin="lower", extent=ex_hr, cmap="Blues", norm=norm)
        axes[1].set_title("Observé")
        fig.colorbar(im0, ax=axes.ravel().tolist(), shrink=0.7)
        fig.suptitle(title_main)
        plt.tight_layout()
        if cap:
            fig.supxlabel(cap, fontsize=9)
    return fig



## Inférence démo : cartes (si données test disponibles)


In [ ]:
def convert_sample_to_batch(sample, builder, device):
    lr_seq = sample["lr"]
    seq_len = lr_seq.shape[0]
    lr_nodes_steps = [builder.lr_grid_to_nodes(lr_seq[t]) for t in range(seq_len)]
    lr_tensor = torch.stack(lr_nodes_steps, dim=0)
    dynamic_features = {node_type: lr_nodes_steps[0] for node_type in builder.dynamic_node_types}
    hetero = builder.prepare_step_data(dynamic_features).to(device)
    out = {"lr": lr_tensor, "residual": sample["residual"], "baseline": sample.get("baseline"), "hetero": hetero}
    if "valid_mask" in sample:
        out["valid_mask"] = sample["valid_mask"]
    return out


def extract_target_and_baseline_t_mean(batch, device):
    target_residual = batch["residual"][-1].to(device)
    baseline_tensor = batch["baseline"][-1] if batch["baseline"] is not None else torch.zeros_like(target_residual)
    baseline_tensor = baseline_tensor.to(device)
    full_target = baseline_tensor + target_residual
    baseline_tensor = torch.nan_to_num(baseline_tensor, nan=0.0, posinf=0.0, neginf=0.0)
    full_target = torch.nan_to_num(full_target, nan=0.0, posinf=0.0, neginf=0.0)
    b = baseline_tensor.unsqueeze(0) if baseline_tensor.dim() == 3 else baseline_tensor
    t = full_target.unsqueeze(0) if full_target.dim() == 3 else full_target
    return t, b


@torch.no_grad()
def run_inference_with_dag(sample):
    batch = convert_sample_to_batch(sample, builder, DEVICE)
    lr_data = batch["lr"].to(DEVICE)
    target_batch, baseline_batch = extract_target_and_baseline_t_mean(batch, DEVICE)
    H_init = encoder.init_state(batch["hetero"]).to(DEVICE)
    drivers = [lr_data[t] for t in range(lr_data.shape[0])]
    seq_output = rcn_runner.run(H_init, drivers, reconstruction_sources=None)
    conditioning = encoder.project_state_tensor(seq_output.states[-1]).to(DEVICE)
    dag_last = seq_output.dag_matrices[-1].detach().cpu()
    samples_out = []
    for _ in range(EVAL_NUM_SAMPLES):
        generated = diffusion.sample(
            conditioning,
            num_steps=EVAL_NUM_STEPS,
            scheduler_type=EVAL_SCHEDULER,
            apply_constraints=False,
            baseline=baseline_batch,
        )
        if generated.t_mean.shape != target_batch.shape:
            generated.t_mean = F.interpolate(
                generated.t_mean, size=target_batch.shape[-2:], mode="bicubic", align_corners=False
            ).clamp(min=0.0)
        generated.t_mean = torch.nan_to_num(generated.t_mean, nan=0.0, posinf=0.0, neginf=0.0)
        samples_out.append(generated)
    return samples_out, target_batch.cpu(), baseline_batch.cpu(), dag_last


pred_mean = tgt = None
dag_last = None
demo_pipe, demo_ds = None, None

if HAS_TEST_DATA and builder is not None:
    try:
        demo_pipe = build_test_pipeline(TEST_GCMS[0])
        demo_ds = demo_pipe.build_sequence_dataset(seq_len=SEQ_LEN, as_torch=True)
        demo_sample = next(iter(demo_ds))
        samples_out, target_batch, baseline_batch, dag_last = run_inference_with_dag(demo_sample)
        pred_stack = torch.stack([s.t_mean for s in samples_out], dim=0).cpu()
        pred_mean = pred_stack.mean(dim=0).squeeze(0).numpy()
        pred_std = pred_stack.std(dim=0).squeeze(0).numpy()
        tgt = target_batch.squeeze(0).numpy()
        if tgt.ndim == 3:
            tgt = tgt[0]
        if pred_mean.ndim == 3:
            pred_mean = pred_mean[0]
            pred_std = pred_std[0]
        err = np.abs(pred_mean - tgt)
        if HAS_CARTOPY:
            lon1d, lat1d = lon_lat_1d_from_pipe(demo_pipe, "hr")
            tgt_a = align_raster_lower_origin(tgt, demo_pipe, "hr")
            pred_a = align_raster_lower_origin(pred_mean, demo_pipe, "hr")
            err_a = align_raster_lower_origin(err, demo_pipe, "hr")
            std_a = align_raster_lower_origin(pred_std, demo_pipe, "hr")
            da_t = numpy_field_to_dataarray(tgt_a, lon1d, lat1d)
            da_p = numpy_field_to_dataarray(pred_a, lon1d, lat1d)
            da_e = numpy_field_to_dataarray(err_a, lon1d, lat1d)
            da_s = numpy_field_to_dataarray(std_a, lon1d, lat1d)
            fig, axes = plt.subplots(
                2,
                2,
                figsize=(10, 9),
                subplot_kw={"projection": ccrs.PlateCarree(central_longitude=171.77)},
                layout="constrained",
            )
            panels = [
                (da_t, "Blues", "Cible HR"),
                (da_p, "Blues", "Prédiction moyenne (ensemble)"),
                (da_e, "hot", "Erreur absolue"),
                (da_s, "viridis", "Écart-type intra-ensemble"),
            ]
            for ax, (da, cmap, tit) in zip(axes.ravel(), panels):
                plot_field_on_map(da, ax, ccrs_m, cmap, robust=False)
                set_map_extent_from_data(ax, ccrs_m, lon1d, lat1d)
                ax.set_title(tit)
            plt.suptitle(f"GCM={TEST_GCMS[0]}")
            plt.show()
        else:
            ex_hr = extent_hr(demo_pipe)
            tgt_a = align_raster_lower_origin(tgt, demo_pipe, "hr")
            pred_a = align_raster_lower_origin(pred_mean, demo_pipe, "hr")
            err_a = align_raster_lower_origin(err, demo_pipe, "hr")
            std_a = align_raster_lower_origin(pred_std, demo_pipe, "hr")
            fig, axes = plt.subplots(2, 2, figsize=(10, 9))
            axes[0, 0].imshow(tgt_a, origin="lower", extent=ex_hr, cmap="Blues")
            axes[0, 0].set_title("Cible HR")
            axes[0, 1].imshow(pred_a, origin="lower", extent=ex_hr, cmap="Blues")
            axes[0, 1].set_title("Prédiction moyenne (ensemble)")
            axes[1, 0].imshow(err_a, origin="lower", extent=ex_hr, cmap="hot")
            axes[1, 0].set_title("Erreur absolue")
            axes[1, 1].imshow(std_a, origin="lower", extent=ex_hr, cmap="viridis")
            axes[1, 1].set_title("Écart-type intra-ensemble")
            for ax in axes.ravel():
                ax.set_xticks([])
                ax.set_yticks([])
            plt.suptitle(f"GCM={TEST_GCMS[0]}")
            plt.tight_layout()
            plt.show()
    except Exception as e:
        print("Inférence démo impossible:", e)
else:
    print("Pas de données test — cartes ignorées.")



### Explicabilité locale (MVP)

D’après [`explicabilite.md`](explicabilite.md) : \(Y_{HR} = \mathrm{Baseline} + \mathrm{Résidu}\). Pour chaque cas ci‑dessous : (1) cartes **Baseline / résidu observé / correction prédite** (prédiction − baseline) ; (2) **heatmap du DAG** instantané du RCN (dernier pas). Une seule inférence par cas (`hr_prediction_explain_bundle`). Les méthodes Captum / DAAM / SHAP du rapport complet ne sont pas branchées ici.


In [ ]:
# Cas présent 1 — CASE_SKIPS_PRESENT[0]
if HAS_TEST_DATA and builder is not None and demo_ds is not None and demo_pipe is not None:
    try:
        sample, _ = nth_sample(demo_ds, CASE_SKIPS_PRESENT[0])
        if sample is None:
            print("Pas assez d'échantillons pour ce skip (CASE_SKIPS_PRESENT[0]).")
        else:
            bundle = hr_prediction_explain_bundle(sample)
            pred_c, tgt_c = bundle["pred"], bundle["tgt"]
            lr_nm = LR_VARIABLES[0] if LR_VARIABLES else None
            ts = sample["time"][-1]
            fig = plot_row_lr_hr_pred(
                demo_pipe,
                sample,
                pred_c,
                tgt_c,
                lr_name=lr_nm,
                gcm_name=TEST_GCMS[0] if TEST_GCMS else None,
                figure_title=f"Cas présent 1 — GCM={TEST_GCMS[0]} — t_fin={ts}",
            )
            plt.show()
            plot_residual_decomposition(
                demo_pipe,
                bundle["baseline"],
                bundle["residual_true"],
                bundle["pred_correction"],
                f"Explicabilité (résidu) — présent 1 — t_fin={ts}",
                sample=sample,
                gcm_name=TEST_GCMS[0] if TEST_GCMS else None,
            )
            plt.show()
            plot_dag_explain(
                bundle["dag_last"],
                f"présent 1 — t_fin={ts}",
                sample=sample,
                gcm_name=TEST_GCMS[0] if TEST_GCMS else None,
            )
            plt.show()
    except Exception as e:
        print("Cas présent 1:", e)
else:
    print("Données test ou demo_ds indisponible.")



In [ ]:
# Cas présent 2 — CASE_SKIPS_PRESENT[1]
if HAS_TEST_DATA and builder is not None and demo_ds is not None and demo_pipe is not None:
    try:
        sample, _ = nth_sample(demo_ds, CASE_SKIPS_PRESENT[1])
        if sample is None:
            print("Pas assez d'échantillons pour ce skip (CASE_SKIPS_PRESENT[1]).")
        else:
            bundle = hr_prediction_explain_bundle(sample)
            pred_c, tgt_c = bundle["pred"], bundle["tgt"]
            lr_nm = LR_VARIABLES[0] if LR_VARIABLES else None
            ts = sample["time"][-1]
            fig = plot_row_lr_hr_pred(
                demo_pipe,
                sample,
                pred_c,
                tgt_c,
                lr_name=lr_nm,
                gcm_name=TEST_GCMS[0] if TEST_GCMS else None,
                figure_title=f"Cas présent 2 — GCM={TEST_GCMS[0]} — t_fin={ts}",
            )
            plt.show()
            plot_residual_decomposition(
                demo_pipe,
                bundle["baseline"],
                bundle["residual_true"],
                bundle["pred_correction"],
                f"Explicabilité (résidu) — présent 2 — t_fin={ts}",
                sample=sample,
                gcm_name=TEST_GCMS[0] if TEST_GCMS else None,
            )
            plt.show()
            plot_dag_explain(
                bundle["dag_last"],
                f"présent 2 — t_fin={ts}",
                sample=sample,
                gcm_name=TEST_GCMS[0] if TEST_GCMS else None,
            )
            plt.show()
    except Exception as e:
        print("Cas présent 2:", e)
else:
    print("Données test ou demo_ds indisponible.")



In [ ]:
# Cas présent 3 — CASE_SKIPS_PRESENT[2]
if HAS_TEST_DATA and builder is not None and demo_ds is not None and demo_pipe is not None:
    try:
        sample, _ = nth_sample(demo_ds, CASE_SKIPS_PRESENT[2])
        if sample is None:
            print("Pas assez d'échantillons pour ce skip (CASE_SKIPS_PRESENT[2]).")
        else:
            bundle = hr_prediction_explain_bundle(sample)
            pred_c, tgt_c = bundle["pred"], bundle["tgt"]
            lr_nm = LR_VARIABLES[0] if LR_VARIABLES else None
            ts = sample["time"][-1]
            fig = plot_row_lr_hr_pred(
                demo_pipe,
                sample,
                pred_c,
                tgt_c,
                lr_name=lr_nm,
                gcm_name=TEST_GCMS[0] if TEST_GCMS else None,
                figure_title=f"Cas présent 3 — GCM={TEST_GCMS[0]} — t_fin={ts}",
            )
            plt.show()
            plot_residual_decomposition(
                demo_pipe,
                bundle["baseline"],
                bundle["residual_true"],
                bundle["pred_correction"],
                f"Explicabilité (résidu) — présent 3 — t_fin={ts}",
                sample=sample,
                gcm_name=TEST_GCMS[0] if TEST_GCMS else None,
            )
            plt.show()
            plot_dag_explain(
                bundle["dag_last"],
                f"présent 3 — t_fin={ts}",
                sample=sample,
                gcm_name=TEST_GCMS[0] if TEST_GCMS else None,
            )
            plt.show()
    except Exception as e:
        print("Cas présent 3:", e)
else:
    print("Données test ou demo_ds indisponible.")



## Prévision : prédit vs observé (fenêtre plus tardive)

Chaque figure compare la **prédiction** et l’**observation HR** au **dernier pas** d’une fenêtre temporelle. Les cas ci‑dessous utilisent des indices `FUTURE_SKIPS` plus grands que les cas « présents », donc un **instant calendaire plus tardif** dans la série test (stride 1). Ce n’est pas une prévision « lead‑1 sans forçage LR futur » : le modèle utilise toute la séquence LR de la fenêtre, comme à l’entraînement.


In [ ]:
# Cas futur 1 — FUTURE_SKIPS[0]
if HAS_TEST_DATA and builder is not None and demo_ds is not None and demo_pipe is not None:
    try:
        sample, _ = nth_sample(demo_ds, FUTURE_SKIPS[0])
        if sample is None:
            print("Pas assez d'échantillons pour ce skip (FUTURE_SKIPS[0]). Réduire FUTURE_SKIPS dans la cellule helpers.")
        else:
            bundle = hr_prediction_explain_bundle(sample)
            pred_c, tgt_c = bundle["pred"], bundle["tgt"]
            ts = sample["time"][-1]
            fig = plot_pred_vs_obs(
                demo_pipe,
                pred_c,
                tgt_c,
                f"Cas futur 1 — GCM={TEST_GCMS[0]} — t_fin={ts}",
                sample=sample,
                gcm_name=TEST_GCMS[0] if TEST_GCMS else None,
            )
            plt.show()
            plot_residual_decomposition(
                demo_pipe,
                bundle["baseline"],
                bundle["residual_true"],
                bundle["pred_correction"],
                f"Explicabilité (résidu) — futur 1 — t_fin={ts}",
                sample=sample,
                gcm_name=TEST_GCMS[0] if TEST_GCMS else None,
            )
            plt.show()
            plot_dag_explain(
                bundle["dag_last"],
                f"futur 1 — t_fin={ts}",
                sample=sample,
                gcm_name=TEST_GCMS[0] if TEST_GCMS else None,
            )
            plt.show()
    except Exception as e:
        print("Cas futur 1:", e)
else:
    print("Données test ou demo_ds indisponible.")



In [ ]:
# Cas futur 2 — FUTURE_SKIPS[1]
if HAS_TEST_DATA and builder is not None and demo_ds is not None and demo_pipe is not None:
    try:
        sample, _ = nth_sample(demo_ds, FUTURE_SKIPS[1])
        if sample is None:
            print("Pas assez d'échantillons pour ce skip (FUTURE_SKIPS[1]). Réduire FUTURE_SKIPS dans la cellule helpers.")
        else:
            bundle = hr_prediction_explain_bundle(sample)
            pred_c, tgt_c = bundle["pred"], bundle["tgt"]
            ts = sample["time"][-1]
            fig = plot_pred_vs_obs(
                demo_pipe,
                pred_c,
                tgt_c,
                f"Cas futur 2 — GCM={TEST_GCMS[0]} — t_fin={ts}",
                sample=sample,
                gcm_name=TEST_GCMS[0] if TEST_GCMS else None,
            )
            plt.show()
            plot_residual_decomposition(
                demo_pipe,
                bundle["baseline"],
                bundle["residual_true"],
                bundle["pred_correction"],
                f"Explicabilité (résidu) — futur 2 — t_fin={ts}",
                sample=sample,
                gcm_name=TEST_GCMS[0] if TEST_GCMS else None,
            )
            plt.show()
            plot_dag_explain(
                bundle["dag_last"],
                f"futur 2 — t_fin={ts}",
                sample=sample,
                gcm_name=TEST_GCMS[0] if TEST_GCMS else None,
            )
            plt.show()
    except Exception as e:
        print("Cas futur 2:", e)
else:
    print("Données test ou demo_ds indisponible.")



In [ ]:
# Cas futur 3 — FUTURE_SKIPS[2]
if HAS_TEST_DATA and builder is not None and demo_ds is not None and demo_pipe is not None:
    try:
        sample, _ = nth_sample(demo_ds, FUTURE_SKIPS[2])
        if sample is None:
            print("Pas assez d'échantillons pour ce skip (FUTURE_SKIPS[2]). Réduire FUTURE_SKIPS dans la cellule helpers.")
        else:
            bundle = hr_prediction_explain_bundle(sample)
            pred_c, tgt_c = bundle["pred"], bundle["tgt"]
            ts = sample["time"][-1]
            fig = plot_pred_vs_obs(
                demo_pipe,
                pred_c,
                tgt_c,
                f"Cas futur 3 — GCM={TEST_GCMS[0]} — t_fin={ts}",
                sample=sample,
                gcm_name=TEST_GCMS[0] if TEST_GCMS else None,
            )
            plt.show()
            plot_residual_decomposition(
                demo_pipe,
                bundle["baseline"],
                bundle["residual_true"],
                bundle["pred_correction"],
                f"Explicabilité (résidu) — futur 3 — t_fin={ts}",
                sample=sample,
                gcm_name=TEST_GCMS[0] if TEST_GCMS else None,
            )
            plt.show()
            plot_dag_explain(
                bundle["dag_last"],
                f"futur 3 — t_fin={ts}",
                sample=sample,
                gcm_name=TEST_GCMS[0] if TEST_GCMS else None,
            )
            plt.show()
    except Exception as e:
        print("Cas futur 3:", e)
else:
    print("Données test ou demo_ds indisponible.")



## Histogrammes et QQ


In [ ]:
if pred_mean is not None and tgt is not None:
    p = pred_mean.ravel()
    t = tgt.ravel()
    mask = np.isfinite(p) & np.isfinite(t)
    p, t = p[mask], t[mask]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    ax1.hist(t, bins=80, density=True, alpha=0.5, label="Cible")
    ax1.hist(p, bins=80, density=True, alpha=0.5, label="Préd. moyenne")
    ax1.set_yscale("log")
    ax1.legend()
    ax1.set_title("PDF (axe Y log)")
    q = np.linspace(0.01, 0.99, 50)
    ax2.scatter(np.quantile(t, q), np.quantile(p, q), s=8, alpha=0.7)
    mn, mx = min(t.min(), p.min()), max(t.max(), p.max())
    ax2.plot([mn, mx], [mn, mx], "k--", lw=1)
    ax2.set_xlabel("Quantiles cible")
    ax2.set_ylabel("Quantiles prédiction")
    ax2.set_title("QQ-plot")
    plt.tight_layout()
    plt.show()


## PSD radiale (approximation)


In [ ]:
def radial_psd(field2d):
    f = np.fft.fftshift(np.fft.fft2(field2d))
    p = np.abs(f) ** 2
    h, w = p.shape
    cy, cx = h // 2, w // 2
    y, x = np.ogrid[:h, :w]
    r = np.sqrt((x - cx) ** 2 + (y - cy) ** 2).astype(int)
    rmax = np.bincount(r.ravel(), weights=p.ravel())
    counts = np.bincount(r.ravel())
    return rmax / np.maximum(counts, 1)


if pred_mean is not None and tgt is not None:
    k1 = radial_psd(tgt)
    k2 = radial_psd(pred_mean)
    plt.figure(figsize=(7, 4))
    plt.semilogy(k1[: min(120, len(k1))], label="Cible")
    plt.semilogy(k2[: min(120, len(k2))], label="Prédiction")
    plt.xlabel("Rayon spectral (bins)")
    plt.ylabel("Énergie")
    plt.legend()
    plt.title("PSD radiale")
    plt.tight_layout()
    plt.show()


## Matrice DAG


In [ ]:
if dag_last is not None:
    A = dag_last
    if A.dim() == 3:
        A = A.mean(dim=0)
    A_np = A.numpy()
    names = VAR_NAMES_DAG if len(VAR_NAMES_DAG) == A_np.shape[0] else [f"V{i}" for i in range(A_np.shape[0])]
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(A_np, xticklabels=names, yticklabels=names, cmap="coolwarm", center=0, ax=ax)
    ax.set_title("DAG (dernier pas RCN)")
    plt.tight_layout()
    plt.show()
else:
    print("DAG non disponible.")


## Cartopy — rappel

Les figures **Cartes par cas** et **Prévision** ci‑dessus utilisent déjà Cartopy (si installé) avec un **extent dérivé** des coordonnées LR/HR du pipeline. Aucun extent géographique fixe n’est requis ici.


In [ ]:
if "HAS_CARTOPY" not in dir():
    HAS_CARTOPY = False
if not HAS_CARTOPY:
    print("Installez cartopy pour les projections géographiques dans les sections précédentes.")
else:
    print("Cartopy disponible — voir cellules « Cartes par cas » et « Prévision ».")


## Synthèse

- Métriques : préférer `results/validation/` après exécution de `st_cdgm_validation_inference.ipynb`.
- Échelle des champs : souvent `log1p` (pr) — voir `config/training_config.yaml`.
- Article : `docs/research_article_st_cdgm.md`.
